In [1]:
import os
from pathlib import Path

import pandas as pd
import numpy as np


In [5]:
DIRECTORY = Path(r'C:\Users\Admin\Desktop\Contamination-Detection\logging_file')
DATA_PATH = DIRECTORY / 'domain_addition_data'

file_list = list(DATA_PATH.glob('*.csv'))
file_list

[WindowsPath('C:/Users/Admin/Desktop/Contamination-Detection/logging_file/domain_addition_data/gemma-2-2b+domain_addition_mask_wrong_answer.csv'),
 WindowsPath('C:/Users/Admin/Desktop/Contamination-Detection/logging_file/domain_addition_data/gemma-2-9b-it+domain_addition_mask_wrong_answer.csv'),
 WindowsPath('C:/Users/Admin/Desktop/Contamination-Detection/logging_file/domain_addition_data/Meta-Llama-3.1-8B-Instruct+domain_addition_mask_wrong_answer.csv'),
 WindowsPath('C:/Users/Admin/Desktop/Contamination-Detection/logging_file/domain_addition_data/Phi-3.5-mini-instruct+domain_addition_mask_wrong_answer.csv'),
 WindowsPath('C:/Users/Admin/Desktop/Contamination-Detection/logging_file/domain_addition_data/Qwen2-7B-Instruct+domain_addition_mask_wrong_answer.csv'),
 WindowsPath('C:/Users/Admin/Desktop/Contamination-Detection/logging_file/domain_addition_data/Vistral-7B-Chat+domain_addition_mask_wrong_answer.csv')]

In [9]:
import re
import string

def normalize_text(text):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    
    return text

def lcs(pred, ref):
    len_pred = len(pred)
    len_ref = len(ref)
    
    dp = [[0] * (len_ref + 1) for _ in range(len_pred + 1)]
    
    for i in range(1, len_pred + 1):
        for j in range(1, len_ref + 1):
            if pred[i - 1] == ref[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    
    return dp[len_pred][len_ref]

def rouge_l_score(pred, ref):
    # normalize
    pred = normalize_text(pred)
    ref = normalize_text(ref)
    pred_tokens = pred.split()
    ref_tokens = ref.split()
    
    # get lcs
    lcs_length = lcs(pred_tokens, ref_tokens)
    
    # precision & recall
    precision = lcs_length / len(pred_tokens) if len(pred_tokens) > 0 else 0
    recall = lcs_length / len(ref_tokens) if len(ref_tokens) > 0 else 0
    
    f1_score = ((2 * precision * recall) / (precision + recall)) if (precision + recall != 0) else 0.0
    
    return f1_score

text1 = 'Tôi tên là Việt Anh'
text2 = 'họ và tên'
rouge_l_score(text1, text2)

0.25

In [21]:
math_syntax_pattern = r'[+\-*/=]'

for file in file_list:
    df = pd.read_csv(file)
    df['label'] = df['label'].astype(str)
    df['score'] = df.apply(lambda row: rouge_l_score(str(row['predict']), str(row['label'])), axis=1)
    df = df.drop(columns=df.columns[df.columns.str.contains('Unnamed')])
    df.to_csv(file, index=False)

    # file_name = os.path.splitext(os.path.basename(file))[0]
    # file_name = file_name.split('_')[0]

    # filtered_df = df[~df['label'].str.contains(math_syntax_pattern, regex=True)]
    # mean_score = filtered_df['score'].mean()

    # print(file_name, mean_score)

In [35]:
df = pd.read_csv(file_list[5])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2283 entries, 0 to 2282
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Question     2283 non-null   object 
 1   predict      639 non-null    object 
 2   label        2283 non-null   object 
 3   rouge_score  2283 non-null   float64
 4   score        2283 non-null   float64
dtypes: float64(2), object(3)
memory usage: 89.3+ KB
